# DIM_ESPECIALIDADE

Especialidade do leito da internação (campo `ESPEC` do SIH), com a descrição oficial
da tabela `LEITOS.CNV` do TAB_SIH (ex.: `01` CIRÚRGICO, `03` CLÍNICO, `07` PEDIÁTRICOS).

**Estratégia de carga:** incremental com `MERGE` (SCD tipo 1). Registros novos recebem a surrogate key da sequence; registros existentes só são atualizados se algum atributo mudou. Reexecutar a carga não duplica linhas nem troca as chaves.

**Membro desconhecido:** `SK = -1` ("NAO INFORMADO"), usado pelas fatos quando o código de origem não existe na dimensão.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import mesclar, executar, consultar

In [ ]:
MEMBRO_DESCONHECIDO = """
MERGE INTO DIM_ESPECIALIDADE d
USING (SELECT -1 AS SK FROM DUAL) s ON (d.SK_ESPECIALIDADE = s.SK)
WHEN NOT MATCHED THEN INSERT (SK_ESPECIALIDADE, COD_ESPECIALIDADE, NOME_ESPECIALIDADE)
VALUES (-1, '-1', 'NAO INFORMADO')
"""
executar(MEMBRO_DESCONHECIDO)

In [ ]:
SQL = """
MERGE INTO DIM_ESPECIALIDADE d
USING (
    SELECT c.CODIGO AS COD_ESPECIALIDADE,
           NVL(t.DESCRICAO, 'NAO CADASTRADA - ' || c.CODIGO) AS NOME_ESPECIALIDADE
    FROM (SELECT CODIGO FROM SILVER_DOMINIO WHERE TABELA = 'ESPECIALIDADE'
          UNION SELECT COD_ESPECIALIDADE FROM SILVER_SIH WHERE COD_ESPECIALIDADE IS NOT NULL) c
    LEFT JOIN SILVER_DOMINIO t ON t.TABELA = 'ESPECIALIDADE' AND t.CODIGO = c.CODIGO
) s
ON (d.COD_ESPECIALIDADE = s.COD_ESPECIALIDADE)
WHEN MATCHED THEN UPDATE SET d.NOME_ESPECIALIDADE = s.NOME_ESPECIALIDADE
  WHERE d.NOME_ESPECIALIDADE <> s.NOME_ESPECIALIDADE
WHEN NOT MATCHED THEN INSERT (SK_ESPECIALIDADE, COD_ESPECIALIDADE, NOME_ESPECIALIDADE)
VALUES (SEQ_DIM_ESPECIALIDADE.NEXTVAL, s.COD_ESPECIALIDADE, s.NOME_ESPECIALIDADE)
"""

In [ ]:
mesclar("DIM_ESPECIALIDADE", SQL, origem="(SELECT CODIGO FROM SILVER_DOMINIO WHERE TABELA = 'ESPECIALIDADE')")

In [ ]:
consultar("SELECT * FROM DIM_ESPECIALIDADE ORDER BY 1 FETCH FIRST 10 ROWS ONLY")